<h1><center>Let's exploit pickle</center></h1>
<h2><center>and skops to the rescue!</center></h2>
<h3><center>Adrin Jalali</center></h3>
<h4><center>probabl.ai</center></h3>
<h4><center>github.com/adrinjalali</center></h3>

In [ ]:
import pickle

In [ ]:
pickle.loads(b"cos\nsystem\n(S'echo hello world'\ntR.")

That relies on `os` being available, which we can customize when loading a pickle file:


``` python
class RestrictedUnpickler(pickle.Unpickler):

    def find_class(self, module, name):
        # Only allow safe classes from builtins.
        if module == "builtins" and name in safe_builtins:
            return getattr(builtins, name)
        # Forbid everything else.
        raise pickle.UnpicklingError(
            f"global '{module}.{name}' is forbidden"
        )
        
with open("file.pkl", "rb") as f:
    obj = RestrictedUnpickler(f).load()
```

*Exploits*: https://ctftime.org/writeup/16723

## PEP 307 - Extensions to the pickle protocol

https://peps.python.org/pep-0307/#security-issues
    
<div>
<img src="figs/security.png" width="600"/>
</div>

# pickles
- Pickler
- Unpickler
- pickling instruction set (`OP` codes)

# `__getstate__`, `__setstate__`

https://docs.python.org/3/library/pickle.html

In [ ]:
class C:
    def __getstate__(self):
        return {"a": 42}
    
    def __setstate__(self, state):
        for key, value in state.items():
            setattr(self, key, value)
            
obj = pickle.loads(pickle.dumps(C()))
obj.a

In [ ]:
import pickletools
pickletools.dis(pickle.dumps(C()))

In [ ]:
with open("/tmp/dumps/oddpickle.pkl", "wb") as f:
    pickle.dump(C(), f)

with open("/tmp/dumps/oddpickle.pkl", "rb") as f:
    obj = pickle.load(f)
obj

# `__reduce__` 👹
https://docs.python.org/3/library/pickle.html#object.__reduce__

Returns a tuple of up to size 6, the first two mandatory:

- A callable object that will be called to create the initial version of the object.
- A tuple of arguments for the callable object. An empty tuple must be given if the callable does not accept any argument.

In [ ]:
class D:
    def __reduce__(self):
        return (print, ("!!!I SEE YOU!!!",))
    
pickled = pickle.dumps(D())
pickletools.dis(pickled)

In [ ]:
pickle.loads(pickled)

In [ ]:
import os
class E:
    def __reduce__(self):
        return (
            os.system,
            ("""echo "!!!I'm in YOUR SYSTEM!!!" > /tmp/dumps/demo.txt""",),
        )
    
pickled = pickle.dumps(E())
pickletools.dis(pickled)

In [ ]:
pickle.loads(pickled)

# Other attacks
- Denial of service
    - Unhandled exceptions
    - Protocol downgrades
    - pickle bombs
- Weird Machine
    - Unused `OP` codes, such as `DUP`
    - Parser abuse
    - Stack corruption
    
source: https://github.com/moreati/pickle-fuzz

In [ ]:
import ast
import pickle
from fickling.fickle import Pickled
print(ast.dump(Pickled.load(pickle.dumps([1, 2, 3, 4])).ast, indent=4))

In [ ]:
print(ast.dump(Pickled.load(pickle.dumps(E())).ast, indent=4))

In [ ]:
!fickling /tmp/dumps/oddpickle.pkl

In [ ]:
with open("/tmp/dumps/badpickle.pkl", "wb") as f:
    pickle.dump(E(), f)

In [ ]:
!fickling /tmp/dumps/badpickle.pkl

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True)

clf = LogisticRegression(solver="liblinear").fit(X, y)
with open("/tmp/dumps/goodpickle.pkl", "wb") as f:
    pickle.dump(clf, f)

In [ ]:
!fickling /tmp/dumps/goodpickle.pkl

**Fickling**: https://github.com/trailofbits/fickling

# skops
More secure persistence with `skops.io`

https://skops.readthedocs.io/en/stable/persistence.html

In [ ]:
import skops.io as sio
sio.loads(sio.dumps(D()))

In [ ]:
sio.loads(sio.dumps(D()), trusted=[D])

### File content

Let's check dumped files!

In [ ]:
sio.dump(D(), "/tmp/dumps/D.skops")

In [ ]:
sio.dump(C(), "/tmp/dumps/C.skops")

In [ ]:
sio.dump(clf, "/tmp/dumps/lr.skops")

## `numpy.save`

https://numpy.org/doc/stable/reference/generated/numpy.save.html

<div>
<img src="figs/numpy-save.png" width="600"/>
</div>

## `numpy.load`

https://numpy.org/doc/stable/reference/generated/numpy.load.html

<div>
<img src="figs/numpy-load.png" width="600"/>
</div>

In [ ]:
sio.load("/tmp/dumps/lr.skops")

In [ ]:
sio.visualize("/tmp/dumps/C.skops")

In [ ]:
sio.visualize("/tmp/dumps/D.skops")

In [ ]:
sio.visualize("/tmp/dumps/lr.skops", show="all")

In [ ]:
from sklearn.preprocessing import FunctionTransformer

def f(x):
    return x + 1

obj = FunctionTransformer(f)
dumped = sio.dumps(obj)

In [ ]:
sio.loads(dumped)

In [ ]:
sio.get_untrusted_types(data=dumped)

In [ ]:
sio.visualize(dumped, show="all")

In [ ]:
sio.loads(dumped, trusted="__main__.f")

# `skops` format

```
zip file:
    schema.json
    139801436035376.npy
    139803280731088.npy
    139803280731952.npy
    139803280801840.npy
```

## Serializers and Loaders

Default serializer/loader:

- `__new__` to construct.
- `__getstate__` and `__setstate__` to get and set attributes.

Special treatment of
- `dict`, `set`, `list`, `tuple`, `type`, `slice`
- `partial`, methods, and functions
- `numpy` and `scipy` arrays, `ufunc`s, and RNGs
- scikit-learn's C extension types
    - hard coded list of allowed objects
    - some C extension types from non scikit-learn libs
        - supported but not trusted by default
- scikit-learn's custom objects
- C extention types supported if their `__reduce__` returns the same type as the constructor
- Already support xgboost, lightgbm, catboost, etc.

## Loading Process

- Load content into memory w/o constructing any objects
- Check included types/functions against a trusted set
- Construct objects if there's nothing we don't trust/know of
    - this is where `__new__` and `__setstate__` are called

## Is it Safe?

- No code is 100% safe!
    - We're trying to make things safe\[er\]!
-`zip` file vulnerabilities apply
    - zip bomb
- Raised exceptions
- Very large objects

## Notes
- Doesn't support any custom object with mandatory args to `__new__`

## Roadmap
- Trust more safe and commonly used types and functions by default
- Speed improvements: memory mapping ndarrays, etc

## Repo
- https://github.com/skops-dev/skops/issues